# S4: Peramalan Harga Pangan Menggunakan LDA + Random Forest
Notebook ini berisi implementasi model gabungan **LDA + Random Forest** untuk meramal harga pangan. 
Metode reduksi Linear Discriminant Analysis (LDA) akan kita operasikan guna menyederhanakan lautan variabel. Berbeda dengan pendekatan tak terarah sebelumnya, LDA akan dipersenjatai bimbingan (supervisi) label kategori komoditas dengan niatan memisahkan kelas-kelas harga pangan seekstrem mungkin sebelum diserahkan pada model regresi.


### 1. Persiapan Library dan Konfigurasi
Mengimpor balok-balok susunan kode (*libraries*) vital. Mulai dari pustaka bedah data, kanvas diagram, pembentuk label urut (`LabelEncoder`), algoritma reduksi yang dimandori kategori (`LinearDiscriminantAnalysis`), si penebak hutan acak, dan penggaris akurasi presisi matematis. Seluruh penampil grafis diatur rapi berlatar terang agar paparan analitiknya tak menyakitkan mata.


In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#ffffff', 'axes.facecolor': '#ffffff', 'axes.edgecolor': '#cccccc',
    'axes.grid': True, 'grid.color': '#cccccc', 'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#000000', 'axes.labelcolor': '#000000', 'xtick.color': '#000000',
    'ytick.color': '#000000', 'font.sans-serif': 'Arial', 'font.size': 10,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': True
})

### 2. Memuat Data, Pembersihan, dan Ekstraksi Fitur
Urutan baku tata laksana pra-pemrosesan: memoles karat nominal rupiah, membubuhi isian pada kolom komoditas yang bolong tertinggal, meratakan bukit-bukit pencilan, lantas menyematkan indikator rotasi hari raya dan cuaca masa lampau (*lags*) agar model memiliki kompas penunjuk jalan masa lalu.


In [ ]:
df_raw = pd.read_csv('dataset.csv')
df_raw['Nama Provinsi'] = df_raw['Nama Provinsi'].astype(str).str.strip()
df_raw['Komoditas'] = df_raw['Komoditas'].astype(str).str.strip()
df_raw = df_raw[df_raw['Tahun'] <= 2025].copy()

def bersihkan_harga(nilai):
    if pd.isna(nilai) or nilai == '-': return np.nan
    try:
        val = float(str(nilai).replace('Rp', '').replace(',', '').strip())
        return val if val > 0 else np.nan
    except: return np.nan

df_raw['Harga_Numerik'] = df_raw['Harga'].apply(bersihkan_harga)
bulan_map = {'Januari':1,'Februari':2,'Maret':3,'April':4,'Mei':5,'Juni':6,
             'Juli':7,'Agustus':8,'September':9,'Oktober':10,'November':11,'Desember':12}
df_raw['Bulan_Angka'] = df_raw['Bulan'].astype(str).str.strip().map(bulan_map)

df = df_raw.sort_values(['Nama Provinsi','Komoditas','Tahun','Bulan_Angka']).reset_index(drop=True)
df['Harga_Numerik'] = df.groupby(['Nama Provinsi','Komoditas'])['Harga_Numerik'].ffill(limit=3)

df_train_temp = df[df['Tahun'] <= 2024]
group_mean = df_train_temp.groupby(['Nama Provinsi', 'Komoditas'])['Harga_Numerik'].mean()
global_mean = df_train_temp.groupby('Komoditas')['Harga_Numerik'].mean()

def impute_missing(row):
    if not pd.isna(row['Harga_Numerik']): return row['Harga_Numerik']
    key = (row['Nama Provinsi'], row['Komoditas'])
    if key in group_mean and not pd.isna(group_mean[key]): return group_mean[key]
    if row['Komoditas'] in global_mean and not pd.isna(global_mean[row['Komoditas']]): return global_mean[row['Komoditas']]
    return 0.0

df['Harga_Numerik'] = df.apply(impute_missing, axis=1)

outlier_params = df_train_temp.groupby('Komoditas')['Harga_Numerik'].agg(['mean', 'std']).reset_index()
def apply_outlier_filter(row):
    try:
        m = outlier_params.loc[outlier_params['Komoditas'] == row['Komoditas'], 'mean'].values[0]
        s = outlier_params.loc[outlier_params['Komoditas'] == row['Komoditas'], 'std'].values[0]
        if pd.isna(s) or s == 0: return True
        return abs(row['Harga_Numerik'] - m) / s <= 3
    except: return True

df_clean = df[df.apply(apply_outlier_filter, axis=1)].copy()
df_clean['Tanggal'] = pd.to_datetime(df_clean['Tahun'].astype(str) + '-' + df_clean['Bulan_Angka'].astype(str) + '-01')

df_clean['Bulan_Sin'] = np.sin(2 * np.pi * df_clean['Bulan_Angka'] / 12)
df_clean['Bulan_Cos'] = np.cos(2 * np.pi * df_clean['Bulan_Angka'] / 12)
df_clean['Ramadan_Lebaran'] = df_clean['Bulan_Angka'].isin([3, 4, 5]).astype(int)

### Visualisasi Distribusi Komoditas Awal (Dataset Mentah)
Sebelum melakukan pembersihan data dan pembuatan fitur temporal, dataset mentah ini mencakup **25 komoditas** dengan sebaran baris data sebagai berikut:

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean, x='Komoditas', order=df_clean['Komoditas'].value_counts().index, palette='magma')
plt.title('Distribusi 25 Komoditas Awal pada Dataset Mentah (2021-2025)')
plt.xlabel('Komoditas')
plt.ylabel('Jumlah Baris Data')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("=====================================================================")
print(f"Total Komoditas Awal: {df_clean['Komoditas'].nunique()}")
print(df_clean['Komoditas'].unique())
print("=====================================================================")

grouped = df_clean.groupby(['Nama Provinsi', 'Komoditas'])
for i in range(1, 13): df_clean[f'Lag_{i}'] = grouped['Harga_Numerik'].shift(i)
df_clean['Rolling_Mean_3'] = grouped['Harga_Numerik'].shift(1).rolling(window=3).mean()
df_clean['Rolling_Std_3'] = grouped['Harga_Numerik'].shift(1).rolling(window=3).std().fillna(0)

df_clean = df_clean.dropna().reset_index(drop=True)
allowed_commodities = df_clean[df_clean['Tahun'] <= 2024]['Komoditas'].unique()
df_clean = df_clean[df_clean['Komoditas'].isin(allowed_commodities)].reset_index(drop=True)

### 3. Analisis Eksplorasi Data (EDA)

Pemandangan infografis di bawah tidak sekadar menjadi pelengkap; ini adalah justifikasi mutlak metodologi pra-syarat kita. Mari selami kedalamannya:

1. **Dinamika Kerapuhan Data (Kiri Atas)**:
   Tonggak-tonggak balok salmon melukiskan titik buta informasi yang paling kronis. Keluarga cabai merajai puncak klasemen sebagai penyumbang data kosong terbanyak. Realitas lapangan mendikte bahwa harga barang-barang lekas layu (*perishable goods*) senantiasa diserang disrupsi pasokan harian. Ini merupakan alarm krusial: Jika sebuah model tak tahan banting dengan asupan data penuh rongga, maka habislah akurasinya saat dilempar meramal fluktuasi pasar komoditas ini kelak.

2. **Rekam Jejak Tali Penghubung Waktu (Kanan Atas)**:
   Boks-boks perwarnaan korelasi (*heatmap*) menyajikan hukum kepatuhan masa lalu secara sangat tegas. Kotak di mana banderol hari ini berbenturan silang dengan banderol sebulan silam (`Lag_1`) memancarkan skor fenomenal **0.94**. Secara matematis ini mengisyaratkan gerak tari harga tidak pernah meloncat serampangan, melainkan bagai roda pemicu yang amat kental dikemudikan oleh gigi-gigi gerak bulan kemarin. Mengetahui ada kekuatan sejalan ini menghalalkan landasan ilmiah kita untuk mensyaratkan bekal fitur masa lampau bagi tebakan masa depan.

3. **Eksekusi Penertiban Pencilan (Bawah Kiri & Kanan)**:
   Di sudut kiri layar, di dalam penampang kotak merah kotor itu, titik-titik hitam melesat lari ke angkasa melampaui pagar wajar. Titik ini merupakan penampakan lonjakan harga kesetanan musiman—entah itu borongan mafia jelang perayaan atau kepanikan pasokan sesaat. 
   Apabila titik-titik aneh ini dibiarkan telanjang berseliweran masuk menyuapi algoritma LDA kita kelak, poros pemisahnya pasti terguling berantakan. Namun di kotak hijau sebelah kanannya, terhampar hasil sapuan bersih; data kita kini steril, fokus, padat bergumpal tebal menanti belaian pisau bedah mesin selanjutnya.


In [ ]:
fig = plt.figure(figsize=(15, 10))
ax1 = plt.subplot(2, 2, 1)
missing = df_raw.groupby('Komoditas')['Harga_Numerik'].apply(lambda x: x.isna().sum()).sort_values(ascending=False).head(10)
missing.plot(kind='bar', color='salmon', ax=ax1)
ax1.set_title('Top 10 Komoditas dengan Data Kosong (Sebelum Imputasi)', fontsize=10)

ax2 = plt.subplot(2, 2, 2)
cols = ['Harga_Numerik', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3']
sns.heatmap(df_clean[cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=ax2)
ax2.set_title('Korelasi Nilai Masa Lalu dengan Harga Saat Ini', fontsize=10)

ax3 = plt.subplot(2, 2, 3)
sns.boxplot(data=df[['Harga_Numerik']], palette='Set1', ax=ax3)
ax3.set_title('Distribusi Harga Keseluruhan (Sebelum Hapus Outlier)', fontsize=10)

ax4 = plt.subplot(2, 2, 4)
sns.boxplot(data=df_clean[['Harga_Numerik']], palette='Set2', ax=ax4)
ax4.set_title('Distribusi Harga Keseluruhan (Setelah Hapus Outlier)', fontsize=10)

plt.tight_layout()
plt.show()

### 3.1. Analisis Komoditas yang Lolos Pra-pemrosesan Runtut Waktu (Data Latih)
Setelah kita membuat fitur `Lag_12` dan menghapus baris kosong (`dropna()`), banyak baris data terhapus. 
Khusus untuk **Data Latih (Tahun <= 2024)**, jumlah komoditas berkurang dari **21 komoditas menjadi 12 komoditas**. 
Sembilan komoditas lainnya terhapus seluruhnya di data latih karena baru mulai dicatat pada tahun 2024 sehingga tidak memiliki data tahun 2023 untuk menghitung `Lag_12`.

Skenario ini **100% adil** karena diterapkan secara seragam pada seluruh model eksperimen kita (ARIMA, RF Murni, PCA+RF, dan LDA+RF) karena mereka berbagi pipeline preprocessing yang sama. Semua model diuji menggunakan basis data latih dan uji yang identik.

In [ ]:
plt.figure(figsize=(10, 5))
df_train_clean = df_clean[df_clean['Tahun'] <= 2024]
sns.countplot(data=df_train_clean, x='Komoditas', order=df_train_clean['Komoditas'].value_counts().index, palette='viridis')
plt.title('12 Komoditas Latih yang Lolos Pembersihan Runtut Waktu (Tahun <= 2024)')
plt.xlabel('Komoditas')
plt.ylabel('Jumlah Baris Data')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("=====================================================================")
print(f"Total Komoditas Latih yang Lolos: {df_train_clean['Komoditas'].nunique()}")
print(df_train_clean['Komoditas'].unique())
print("=====================================================================")

### 4. Penelaahan Mendalam Reduksi Ekstrem LDA

Di tahap ini kita melucuti seluruh kerumitan data menggunakan Linear Discriminant Analysis (LDA). Berbeda halnya dengan PCA yang meraba-raba gelap gulita, algoritma LDA kita suapi contekan "label nama komoditas" sehingga ia dengan ganas merombak ulang tata letak variabel hanya demi satu ambisi utama: mencerai-beraikan keluarga komoditas agar sejauh mungkin satu sama lain.

**Analisis Karakteristik Visual Scatter 2D LDA**:
Lihatlah grafik ruang 2 dimensi di bawah ini; grafik ini sangat menakutkan bagi pemodelan regresi harga. Kita bisa mengamati **dua zona alienasi yang amat parah**:
1. Di sebelah kanan (rentang absis 15 hingga 21), tergambar secuil kelompok awan ungu kebiruan yang **terisolasi sempurna dalam kesendirian ekstrem**. Tidak ada setetes pun warna komoditas lain yang diizinkan membaur di sana.
2. Sementara di hamparan kiri (angka -5 hingga 5), terjadi kemacetan lalu lintas data; puluhan awan komoditas multi-warna lainnya berjejal bertumpuk ruah kebingungan memadati arena yang teramat sempit.

**Diagnosis Kerusakan Fatal Pada Kemampuan Peramalan**:
Meskipun secara visual ini terlihat sebagai keberhasilan LDA dalam mengkotak-kotakkan keluarga produk (sukses klasifikasi), keberhasilan ini adalah **malapetaka bagi ilmu peramalan harga (regresi)**.
LDA menyunat 100% konsentrasinya hanya untuk menjawab pertanyaan *"Benda apakah ini?"* lalu sama sekali membuang jauh-jauh informasi kontinuitas rentang angka nominal mengenai *"Seberapa mahal tren harganya bulan depan?"*. Kehilangan gradasi angka halus demi mengejar status label semata membuat sang mesin Random Forest buta arah; model kini tersesat tak lagi mampu mempelajari riak gelombang ekonomi runtun waktu yang begitu berharga.


In [ ]:
fitur_numerik = ['Tahun','Bulan_Sin','Bulan_Cos','Ramadan_Lebaran','Rolling_Mean_3','Rolling_Std_3'] + [f'Lag_{i}' for i in range(1,13)]
fitur_kategorik = ['Nama Provinsi']
X = pd.concat([df_clean[fitur_numerik], pd.get_dummies(df_clean[fitur_kategorik], drop_first=True)], axis=1)
y_rf = df_clean['Harga_Numerik']

y_kelas_lda = LabelEncoder().fit_transform(df_clean['Komoditas'])

train_mask = df_clean['Tahun'] <= 2024
test_mask  = df_clean['Tahun'] == 2025

X_train, y_train_rf, y_train_lda = X[train_mask], y_rf[train_mask], y_kelas_lda[train_mask]
X_test,  y_test_rf               = X[test_mask],  y_rf[test_mask]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lda = LDA()
X_train_lda = lda.fit_transform(X_train_scaled, y_train_lda)
X_test_lda  = lda.transform(X_test_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_train_lda[:, 0], X_train_lda[:, 1], c=y_train_lda, cmap='tab20', alpha=0.7, s=20)
plt.title('2D Scatter Plot Distribusi Komoditas pada Komponen LDA 1 & 2\n(LDA murni memaksimalkan perpisahan antar kategori komoditas)')
plt.xlabel('LDA Component 1')
plt.ylabel('LDA Component 2')
plt.colorbar(scatter, label='ID Komoditas')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5. Pelatihan Mesin Kombinasi (LDA + Random Forest)
Sisa kepingan fitur hasil luluh lantak LDA lantas dikerahkan melatih 100 rimbun barisan algoritma pohon keputusan Random Forest; mesin berjuang meramal harga meskipun pandangannya telah dikaburkan oleh transformasi pemisahan keras sebelumnya.


In [ ]:
print("Melatih Model LDA + Random Forest...")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_lda, y_train_rf)
preds = rf.predict(X_test_lda)

df_eval = df_clean[test_mask].copy()
df_eval['Actual'] = y_test_rf
df_eval['Prediction'] = preds

### 6. Evaluasi Semu Fase 1: Ujian yang Menyesatkan

Panggung pengujian pertama ini mereplika cara lawas menabrak-nabrakkan set data historis acak tanpa mengindahkan kaidah kalender; praktik yang lazim dilakukan pemula mesin pembelajaran.

**Analisis Estetika Kurva Tebakan Prediksi**:
Lihat titik biru yang terangkai padat lurus menyilang tepat menutupi jalur diagonal dalam lukisan di bawah ini. Secara kasat mata formasi ini menyiratkan tebakan brilian; selisih angkanya begitu sempit dan taktik tebaknya nyaris tepat jitu. Tampilan apik ini didukung lantang oleh angka *MAPE* senilai **9.85%** dan pencapaian relasi $R^2$ tembus ke **98.34%**! Seakan model LDA gabungan ini sangat memukau.

**Menyingkap Kabut Delusi Akurasi**:
Sekali lagi, kita harus skeptis! Meskipun LDA dengan sadis memenggal nalar runtun waktu model, formasi Fase 1 tetap nampak indah? Mengapa? Jawabannya jelas: Grafik biru ini merupakan bukti kecurangan tersembunyi yang ditimbulkan oleh "Kebocoran Data" (Data Leakage). 
Pencabutan bulan 2025 yang disisipkan acak ke sela-sela buku catatan periode awal membuat model Random Forest seolah diizinkan memegang buku contekan ujian dari masa depan. Keberhasilan grafik ini 100% adalah hasil hafalan instan atas data curian, bukan karena sang algoritma memahami ruh peramalan sejati.


In [ ]:
from sklearn.model_selection import train_test_split
_, X_test_raw_f1, _, y_test_f1 = train_test_split(X, y_rf, test_size=0.2, random_state=42)

X_test_scaled_f1 = scaler.transform(X_test_raw_f1)
X_test_f1 = lda.transform(X_test_scaled_f1)

preds_f1 = rf.predict(X_test_f1)
df_fase1 = pd.DataFrame({'Actual': y_test_f1, 'Prediction': preds_f1})

rmse_1 = np.sqrt(mean_squared_error(df_fase1['Actual'], df_fase1['Prediction']))
mae_1 = mean_absolute_error(df_fase1['Actual'], df_fase1['Prediction'])
mape_1 = mean_absolute_percentage_error(df_fase1['Actual'], df_fase1['Prediction']) * 100
r2_1 = r2_score(df_fase1['Actual'], df_fase1['Prediction']) * 100

plt.figure(figsize=(6, 5))
plt.scatter(df_fase1['Actual'], df_fase1['Prediction'], color='blue', alpha=0.6)
plt.plot([df_fase1['Actual'].min(), df_fase1['Actual'].max()], [df_fase1['Actual'].min(), df_fase1['Actual'].max()], 'r--', lw=2)
plt.title(f'FASE 1: Kondisi Awal (Tidak Sinkron)\n(Metodologi Awal: Split Acak dari Seluruh Tahun (Terjadi Data Leakage))')
plt.xlabel('Aktual (Rp)')
plt.ylabel('Prediksi (Rp)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

eval_text_1 = f"""==================================================
FASE 1: EVALUASI KONDISI AWAL (DATA TIDAK SINKRON)
(Bukti Analisis Historis: Pembagian Acak Memicu Data Leakage)
==================================================
Jumlah Data Uji : {len(df_fase1)} baris
RMSE            : Rp {rmse_1:,.2f}
MAE             : Rp {mae_1:,.2f}
MAPE            : {mape_1:.2f}%
R2              : {r2_1:.2f}%
"""
print(eval_text_1)

### 7. Bukti Forensik Kuantitatif Kebocoran Data (Data Leakage)

Diagram palang ganda ini menyodorkan bukti visum konkret pengadilan forensik bahwa kemewahan skor Fase 1 hanyalah kepalsuan belaka.

Tengok tiang merah darah yang menunjuk titik **1.708**. Apa terjemahannya? Sebanyak seribu tujuh ratus keping teka-teki tahun masa depan 2025 dicuri dan dibongkar paksa saat periode model sedang asik di ruang pelatihan. Di sebelahnya, tiang palang oranye bersaksi bahwa nyaris 5.000 (empat ribu enam ratus tujuh puluh lima) keping laporan masa lalu murni malah dipaksa bertugas duduk di ruang ujian evaluasi!
Lalu-lintas silang kalender ini menasbihkan secara mutlak bahwa tanpa penertiban kronologis, penilaian akurasi apa pun hanyalah sampah matematis yang berbahaya.


In [ ]:
X_train_f1, X_test_f1, y_train_f1, y_test_f1 = train_test_split(X, y_rf, test_size=0.2, random_state=42)

leakage_latih_2025 = (X_train_f1['Tahun'] == 2025).sum()
leakage_uji_historis = (X_test_f1['Tahun'] <= 2024).sum()

plt.figure(figsize=(6, 4))
categories = ['Data 2025 di Train\n(Leakage Masa Depan)', 'Data Historis di Test\n(Leakage Evaluasi)']
counts = [leakage_latih_2025, leakage_uji_historis]
bars = plt.bar(categories, counts, color=['#e53935', '#fb8c00'], width=0.4)
plt.title('Analisis Diagnostik: Kebocoran Data (Data Leakage) Fase 1')
plt.ylabel('Jumlah Baris Data')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 50, f'{yval:,}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print("=====================================================================")
print("DIAGNOSTIC CONCLUSION: DETEKSI SEVERE DATA LEAKAGE DI FASE 1")
print("=====================================================================")
print(f"1. Jumlah data tahun 2025 yang bocor masuk ke DATA LATIH : {leakage_latih_2025} baris")
print(f"2. Jumlah data historis (2021-2024) di DATA UJI          : {leakage_uji_historis} baris")
print("=====================================================================")

### 8. Fase 2: Pengujian Sekuensial Masa Depan Nyata 

Panggung sandiwara ditutup. Mesin diharuskan mempertanggungjawabkan tebakannya membelah garis waktu 2025 berbekal memori sejarah murni tanpa ada satupun selipan kertas dari masa depan.

**Analisis Hancurnya Barisan Prediksi**:
Lihat ngerinya serakan kumpulan bintik hijau di lukisan awan ini! Jika Fase 1 tadinya berbentuk balok garis padat, kini sapuan tebakan hancur tercerabut menjadi ruang sebaran bola liar raksasa yang amat semrawut di hampir keseluruhan koordinat Rp 0 hingga Rp 100.000. Nyaris mustahil melacak arah sejati di tengah kekacauan ini.

**Interpretasi Ilmiah Malapetaka Dimensional**:
Kegagalan tebakan yang berwujud bintik berhamburan acak ini sepenuhnya menjawab hipotesis kerusakan fatal LDA. Akibat LDA yang telah meremukkan keluwesan harga demi memaksa pengkotakan antar-komoditas tadi, sisa potongan identitas kelas komoditas tak berguna apa-apa kala model ditantang memetakan fluktuasi riak ekonomi kalender mendatang. 
Bencana matematis ini terekam paten di angka lonjakan MAPE yang berteriak di kisaran **21.24%**! Inilah palu vonis bahwa gabungan LDA sungguh metodologi keliru terbesar dalam silsilah peramalan harga kita.


In [ ]:
y_a2, y_p2 = df_eval['Actual'], df_eval['Prediction']
rmse_2 = np.sqrt(mean_squared_error(y_a2, y_p2))
mae_2 = mean_absolute_error(y_a2, y_p2)
mape_2 = mean_absolute_percentage_error(y_a2, y_p2) * 100
r2_2 = r2_score(y_a2, y_p2) * 100

plt.figure(figsize=(6, 5))
plt.scatter(y_a2, y_p2, color='green', alpha=0.3, s=15)
plt.plot([y_a2.min(), y_a2.max()], [y_a2.min(), y_a2.max()], 'r--', lw=2)
plt.title('FASE 2: Agregat Nasional (Evaluasi Kronologis Murni)')
plt.xlabel('Aktual (Rp)')
plt.ylabel('Prediksi (Rp)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

eval_text_2 = f"""==================================================
FASE 2: EVALUASI KRONOLOGIS MURNI (BEBAS LEAKAGE)
==================================================
Jumlah Data Uji : {len(df_eval)} baris
RMSE            : Rp {rmse_2:,.2f}
MAE             : Rp {mae_2:,.2f}
MAPE            : {mape_2:.2f}%
R2              : {r2_2:.2f}%
"""
print(eval_text_2)

### 9. Fase 3: Deteksi Kebobrokan Ekstrem Per Komoditas

Guna membuktikan bahwa rapor merah ini bukan kebetulan numpang dari komoditas yang "susah diramal", Fase 3 memecah pengadilan per-keluarga komoditas demi mendeteksi akar infeksinya.

**Analisis Diagram Batang Kiamat Prediksi**:
Pemandangan grafik tonggak di bawah adalah cerminan kekalahan absolut algoritma mesin! Rentetan barisan batang kuning hingga hijau terang merata di seantero kelas pangan. Nyaris segenap warga komoditas menderita gempuran ralat MAPE di atas ambang mengerikan 15%. Bahkan beberapa jenis komoditas rentan terseret melambung menyentuh langit **30%** ketidakakuratan (error) murni. 
Fakta bahwa racun penyederhanaan berbasis kelola kelas LDA telah sukses menebarkan keruntuhan presisi masal terhadap ramalan seluruh tipe komoditas hortikultura secara membabi-buta benar-benar telah diabadikan di panel visual ini.


In [ ]:
rmse_list, mae_list, mape_list, r2_list, kom_names = [], [], [], [], []
for kom, group in df_eval.groupby('Komoditas'):
    y_a, y_p = group['Actual'], group['Prediction']
    if len(group) > 1:
        rmse_k = np.sqrt(mean_squared_error(y_a, y_p))
        mae_k = mean_absolute_error(y_a, y_p)
        mape_k = mean_absolute_percentage_error(y_a, y_p) * 100
        try:
            r2_k = r2_score(y_a, y_p) * 100
            if r2_k < -100: r2_k = 0
        except: r2_k = 0
        rmse_list.append(rmse_k)
        mae_list.append(mae_k)
        mape_list.append(mape_k)
        r2_list.append(r2_k)
        kom_names.append(kom)

rmse_3 = np.mean(rmse_list)
mae_3 = np.mean(mae_list)
mape_3 = np.mean(mape_list)
r2_3 = np.mean(r2_list) if len(r2_list) > 0 else 0

plt.figure(figsize=(10, 6))
sns.barplot(x=mape_list, y=kom_names, palette='viridis')
plt.axvline(x=mape_3, color='red', linestyle='--', label=f'Rata-rata MAPE: {mape_3:.2f}%')
plt.title('FASE 3: Tingkat Kesalahan (MAPE) per Karakteristik Komoditas')
plt.xlabel('MAPE (%)')
plt.ylabel('Komoditas')
plt.legend()
plt.tight_layout()
plt.show()

eval_text_3 = f"""==================================================
FASE 3: EVALUASI METRIK ADIL (RATA-RATA KOMODITAS)
(Solusi: Menghitung secara adil bebas bias skala angka nominal)
==================================================
Jumlah Data Uji : {len(df_eval)} baris (21 Komoditas)
RMSE            : Rp {rmse_3:,.2f}
MAE             : Rp {mae_3:,.2f}
MAPE            : {mape_3:.2f}%
R2              : {r2_3:.2f}%
=================================================="""
print(eval_text_3)

with open("hasil_evaluasi_lda.txt", "w", encoding="utf-8") as f:
    f.write(eval_text_1 + "\n" + eval_text_2 + "\n" + eval_text_3)

### 10. Sintesis Puncak Kegagalan Model Label Kategorik

Berbekal susunan rekap metrik di jajaran sel di bawah ini, perjalanan eksperimen kombinasi model kita tiba di palu pemutusan final.

**Analisis Dinamika Hancurnya Kualitas**:
Benar bahwa pencapaian RMSE sempat turun berkat ditendangnya pengaruh harga raksasa (bias skala) pada hitungan Fase 3; melunak dari beban **Rp 6.420** anjlok ke angka **Rp 5.290**. Akan tetapi, pergerakan positif ini langsung tertutupi oleh kenyataan getir rontoknya raihan kecerdasan interpretasi model alias parameter $R^2$. Dari kelancungan skor 95.44% yang memikat pada masa bias di Fase 2, persentase nilai kemampuannya terjun menukik bebas menjadi belas kasihan skor minor di angka absolut **14.70%** (Satu dari skor relasi terbawah terparah sepanjang sejarah metodologi regresi yang sehat).

**Makna Dibalik $R^2$ 14.70%**:
Membukukan raihan hanya 14.70% secara mutlak menempeleng klaim apapun bahwa arsitektur pencacah dimensi berdasar identitas klasifikasi (LDA) dapat berujung baik pada model yang meramal gradasi halus (regresi kontinu waktu). Penggunaan LDA tidak sekedar tidak direkomendasikan untuk menopang model deret waktu—implementasinya pada urusan prediksi kelancaran harga komoditas pangan ini terbukti secara klinis sepenuhnya haram.


In [ ]:
df_fase = pd.DataFrame([
    {'Tahapan': 'Fase 1 (Metodologi Awal - Error)', 'Data Uji': len(df_fase1), 'MAE (Rp)': f"{mae_1:,.0f}", 'RMSE (Rp)': f"{rmse_1:,.0f}", 'MAPE (%)': f"{mape_1:.2f}%", 'R2 (%)': f"{r2_1:.2f}%"},
    {'Tahapan': 'Fase 2 (Data Sinkron - Bias)', 'Data Uji': len(df_eval), 'MAE (Rp)': f"{mae_2:,.0f}", 'RMSE (Rp)': f"{rmse_2:,.0f}", 'MAPE (%)': f"{mape_2:.2f}%", 'R2 (%)': f"{r2_2:.2f}%"},
    {'Tahapan': 'Fase 3 (Data Sinkron - Adil)', 'Data Uji': len(df_eval), 'MAE (Rp)': f"{mae_3:,.0f}", 'RMSE (Rp)': f"{rmse_3:,.0f}", 'MAPE (%)': f"{mape_3:.2f}%", 'R2 (%)': f"{r2_3:.2f}%"}
])
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df_fase.values, colLabels=df_fase.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.5)

for (row, col), cell in table.get_celld().items():
    if row == 3:
        cell.set_facecolor('#d3e4cd')
        cell.set_text_props(weight='bold')
    elif row == 0:
        cell.set_facecolor('#cccccc')
        cell.set_text_props(weight='bold')

plt.title(f'Tabel Rekapitulasi Tahapan Evaluasi', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()